In [ ]:
Content
1. ACID
    - 1.1 Logs
2. LOCKs
3. Read Phenomenas
4. Isolation Levels
5. Eventual Consistency
6. Connection Pooling

# 1. ACID

In [ ]:
# Transaction
A transaction is a single unit of work that performs one or more operations on the database. These operations are executed in a way that ensures data integrity.
 - One unit of work and it is usually a collection of queries.

Examples:
	•	Transferring money between two accounts. (Select, Update, Update)
	•	Inserting an order and updating inventory.

Transaction Stages:
 BEGIN → (one or more operations) → COMMIT / ROLLBACK
          In memory only

# Pattern
BEGIN TRY
    BEGIN TRANSACTION
                            -- Your queries
    COMMIT TRANSACTION
END TRY
BEGIN CATCH
    IF @@TRANCOUNT > 0   # "Is there any active transaction currently open?"
        ROLLBACK TRANSACTION
    PRINT ERROR_MESSAGE();
END CATCH


Example:

BEGIN TRANSACTION;

BEGIN TRY
    DECLARE @amount INT = 1000;
    DECLARE @sourceId INT = 1;
    DECLARE @destId INT = 2;

    -- 1. Lock both accounts to prevent race conditions (UPDLOCK = exclusive intent)
    -- Use HOLDLOCK to emulate SERIALIZABLE isolation
    SELECT Balance
    FROM Accounts WITH (UPDLOCK, HOLDLOCK)
    WHERE AccountId IN (@sourceId, @destId);

    -- 2. Attempt to deduct balance only if sufficient balance exists
    UPDATE Accounts
    SET Balance = Balance - @amount
    WHERE AccountId = @sourceId AND Balance >= @amount;

    -- 3. Check if deduction was successful
    IF @@ROWCOUNT = 0
    BEGIN
        RAISERROR('Insufficient balance or source account not found.', 16, 1);
    END

    -- 4. Credit to destination
    UPDATE Accounts
    SET Balance = Balance + @amount
    WHERE AccountId = @destId;

    -- 5. Commit
    COMMIT;
    PRINT 'Transaction committed.';
END TRY
BEGIN CATCH
    ROLLBACK;
    PRINT 'Transaction failed.';
    PRINT ERROR_MESSAGE();
END CATCH;


In [ ]:
# ACID
 - ACID is an acronym that guarantees the reliability of database transactions.

# Without ACID
    •	Data can become corrupted (e.g., partial update).
	•	Race conditions can occur (e.g., two users buying the last item).

                                   
1. Atomicity: All operations in a transaction succeed or none do.
	- If one part of the transaction fails, the whole transaction fails.
	- Ensured using rollback if failure occurs.
    - Lack of atomicity, leads to In-consistency.
    - Example: In a money transfer, if money is debited from account A but can’t be credited to account B, the debit must also be undone.

2. Consistency: The database must be in a valid state before and after the transaction.
	- All constraints, cascades, triggers, and rules are respected.
    - Some database sacrifice consistance for speed and performance.
    - Example: If a constraint says balance cannot go negative, a transaction cannot leave a user with a negative balance.
        - Or if balance deducted from one account and not credited in another account

    - Types
        1. Consistency in Data: Guarantees that only valid data is written
            - Enforced by:
                - Constraints (e.g., CHECK, FOREIGN KEY)
                - Business logic
                - Triggers or stored procedures
                - Transactions (via Atomicity)
                - Isolation
        2. Consistency in reads: read consistency is affected by isolation levels



                                   

In [ ]:
3. Isolation: Transactions don’t interfere with each other.
    - Concurrent transactions should not interfere with each other.
	- One transaction should not see uncommitted changes of another.
    - Example: While one transaction is updating a product price, another should not read it until committed.
    - Isolation is managed via:
    	- Locks
    	- Isolation levels (Read Uncommitted, Read Committed, Repeatable Read, Serializable)

4. Durability: Once a transaction is committed, the changes are permanent, even if the system crashes.
	- To ensure durability, databases explicitly flush data to disk, using fsync()
        - DB transaction begin -> Commit
        -  Database writes log and data → OS buffers (not yet on disk)
        - If database is crashed, then the data is lost
    - Without explicite fsync()
        - The OS might eventually write to disk
        - ❌ But the database cannot guarantee durability
        - ❌ If a crash happens before the OS flush, the write is lost
    - Why we don't write it to DB directly: It slows down the write updates.
    - WAL + Flush = Real Durability
        - In durable databases:
            - Transaction is logged in WAL
            - WAL is flushed with fsync()
            - Only after flush returns, the transaction is considered committed
    - Ensured via write-ahead logs or redo logs.
            - You must write the log before writing the actual data.
            - If a transaction modifies data, the database first logs the intended change to a persistent WAL file on disk. Only after that does it apply the change to the database file (in memory or on disk).
     - Asynchronous snapshot

# WAL + Flush = Real Durability
Step 0: Client sends COMMIT Request
Step 1: Modify data page (in memory)
Step 2: Create WAL entry for commit  (WAL entry creation = in-memory RAM operation/ WAL buffer)
Step 3: Flush WAL to disk (fsync) ← durability happens here
    - fsync(): Forces OS to write data from memory → actual disk
    - fsync() writes WAL (log), NOT the actual data pages
Step 4: DB sends COMMIT success to client
Step 5: Data page still in memory (dirty page)
    - DB will return the UPDATED value (from memory), even if the data page is not yet written to disk.
Step 6: Later → written to disk
    - Why not write immidiately, Disk I/O operations are expensive, so, it performs writing in Batches later.

### 1.1 Logs

In [ ]:
# Logs
WAL, Redo logs, and Undo logs are core to how databases guarantee ACID properties, especially durability and atomicity.
All SQL DB use this - SQL server, My sql, Postgress

1.  WAL (Write-Ahead Logging)
    - A protocol where all changes are written to a log before being applied to the database. This ensures that if the database crashes, you can replay the log to restore committed transactions.
    - Rule: Write to log → Flush log to disk → Apply change to data files
    - Purpose:
    	•	Guarantees durability — a committed transaction won’t be lost even if the DB crashes.
    	•	Allows fast recovery after a crash.    
    - Working:
    	1.	UPDATE Salary from 50k → 60k.
    	2.	DB writes (TransactionID=123, UPDATE Salary, Old=50k, New=60k) to WAL.
    	3.	WAL is flushed to disk.
    	4.	DB updates the data file in memory and eventually flushes it to disk.
    - Steps Involved
        1. Lets say we executed a query UPDATE employees SET salary = 60000 WHERE id = 123;
        2. Modify in memeory
            •	The database loads the relevant page containing employee id=123 into its buffer pool (RAM).
        	•	It changes the salary in memory — but does not yet write it to the main data file.
        3. Write to WAL
            •	A log entry is created
            •	Append change record, either to memory or to WAL file (possibly still in OS cache). (sequential write = fast).
            •	Query: TX123: page_id=42, old_salary=50000, new_salary=60000

            •	Meaning: The database prepares a log record describing the change (which row, old value, new value, etc.) and places it in memory inside the WAL buffer (also called log buffer).
        	•	Where it is stored: In RAM, not yet on permanent storage.
        	•	Why: Writing to memory is very fast, and we can accumulate multiple WAL entries before flushing.

        4. Flush WAL to disk (ensure durability)
            •	The WAL must be physically flushed to storage before the DB can acknowledge COMMIT to the client.
        	•	This ensures that even if the server crashes immediately after commit, we still have the instructions to reapply the change.

        	•	Meaning: The DB physically writes the WAL buffer contents to the WAL file on persistent storage (SSD/HDD).
        	•	This is the point of durability: Once the WAL entry is on disk, it survives a crash or restart.
        	•	How: Usually done by calling the OS fsync() or fdatasync() to guarantee the data is actually stored on disk, not just sitting in the OS page cache.
        5. DB updates in-memory copy of the data
        6. Commit ACK is sent to the user ✅
        7.  Apply change to data file later (Background process)
        	•	The in-memory page with the new salary will eventually be written (“flushed”) back to the main data file — but this may happen much later (seconds or minutes).
        	•	This process is handled by a background checkpoint process, not at transaction commit time.   
            •	It sync changes when a checkpoint occurs or The DB decides it’s time to sync changes.
        8. Checkpoint → Mark all changes before a certain LSN (Log Sequence Number) as safe on disk
        9. Truncate / reuse WAL space for those old entries. When database thinks, like No recovery scenario would need that entry.

                               
# Why not update the data file immediately?
	•	Data files can be large and scattered — random disk writes are slow.
	•	WAL writes are small, sequential, append-only, which is extremely efficient.
	•	It allows the DB to batch multiple updates into one disk write for the main data file.

# Crash recovery case
        If DB crashes after WAL is flushed but before the data file was updated.
            - On restart, DB replays the WAL and it re-applies any committed transactions that were not yet reflected in the main files.

•	Rolled-back transaction:
	•	Its WAL entries exist in memory.
	•	They may never reach disk if rollback occurs before commit.
	•	If a crash occurs mid-rollback, recovery can use whatever was in WAL to undo partially applied operations.
# process
[1] UPDATE row in table
     |
     v
[2] Generate WAL record -> Write to WAL file -> Flush WAL (durability)
     |
     v
[3] Update in-memory copy of the table (buffer pool)
     |
     v
[4] Later: Write dirty page from buffer pool -> Table data file



# How databases reduce the risk
	•	Synchronous commits — Don’t confirm COMMIT to the client until WAL is persisted to disk.
    	•	PostgreSQL: synchronous_commit=on ensures fsync at commit.
    	•	MySQL InnoDB: innodb_flush_log_at_trx_commit=1 does fsync on each commit.
	•	Asynchronous commits — Faster, but may lose last few transactions in a crash.
    	•	PostgreSQL: synchronous_commit=off — WAL stays in memory briefly.
    	•	MySQL: innodb_flush_log_at_trx_commit=2 — flush to OS cache but not disk.            

In [ ]:
2. Redo Logs
    •	Logs that store how to re-apply (redo) a change.
    •	They’re used during crash recovery to ensure committed changes are reflected in the database, even if the data pages weren’t flushed yet.
	•	Redo logs are the actual records in the log that describe how to reapply changes to get the DB to the latest committed state.
	•	Redo logs are data, WAL is the rule that forces us to write that data first.
	•	Stored in transaction log files or redo log files on disk.
	•	Used after crash to reapply changes from committed transactions that weren’t fully written to data files yet.
    •	Example scenario:
        •	You commit a transaction, WAL is flushed, but before the updated page is written to disk, the system crashes.
        •	On restart, the redo log says: TransactionID=123 → Set Salary=60k

3. Undo Logs
    •	Logs that store how to reverse (undo) a change. DB uses undo log to restore the original value.
    •	They’re used when:
    	•	A transaction rolls back.
    	•	Crash recovery needs to remove uncommitted changes from the DB.

    •	Purpose:
    	•	Guarantees atomicity — if a transaction fails, all its partial work is removed.
    •	Undo log entry: TransactionID=124 → Set Salary back to 50k



# 2. LOCKs

In [ ]:
Locking Strategies:
    ├── Pessimistic Locking
    │     ├── Shared Lock (S)
    │     ├── Exclusive Lock (X)
    │     ├── Update Lock (U)
    │     ├── Intent Locks (IS, IX)
    │     └── Schema / Key-Range Locks
    └── Optimistic Locking
          ├── Version/Timestamp Check
          └── Retry on Conflict

1. Pessimistic Locking
    - Idea: Assume conflicts will happen, so prevent them
    - Working
    	- When a transaction reads or updates data, it locks the row/table so no other transaction can modify it until the first one finishes.
    	- Prevents conflicts at the cost of reduced concurrency.
    	- Other transactions must wait or fail if they try to access the same resource.
    - Types
        1. Shared (S): Used for reads. Others can also read, but no writes allowed.
            - I want to read a row in a long transaction block, and I do not want any other query to edit/ update the row.
            - Many users can acquire a shared lock, but no one can acquire a exclusive lock.
        2. Update (U): Update locks are used to prevent deadlocks during read-to-write transitions, and are later upgraded to exclusive locks when the actual modification occurs.
        3. Exclusive Lock (X): Used for writes. No one can read or write until it’s released.
            - I want to read/update a row, and I do not want any other can read or update the row.
            - If I want exclusive lock, then no one else can acquire a exclusive/ shared lock.
        4. Intent Locks (IS/IX): Placed at higher levels (e.g., table) to indicate intent to lock rows.
# Example
BEGIN TRANSACTION;
SELECT * FROM Orders WHERE OrderID = 5 FOR UPDATE;          --  FORUPDATE locks the row so no other transaction can change it until commit Exclusive lock (X lock) 
UPDATE Orders SET Status = 'Shipped' WHERE OrderID = 5;
COMMIT;

2. Optimistic Locking (using rowVersion, UpdatedAt)
    - Idea: Assume conflicts are rare, so don’t lock. But detect and resolve if they happen.
    - We don’t use locks during the read phase — but you still need a way to ensure data wasn’t changed by someone else between read and write.
    - Working:
        - Read data without locking.
    	- When updating,  it checks if the data has changed since it was read (using timestamps, version numbers, or hashes).
        	- If it changed, abort or retry the transaction.
    - Example:
        - In Read query, fetch the RowVersion or UpdatedAt time, and while updating the row, check if rowVersion or UpdatedAt at same as we fetched earlier in the same long transaction.
        - If same, then increment these columns, else fail the transaction.
        - Table with row version
            - ALTER TABLE Orders ADD RowVersion ROWVERSION;
            - UPDATE Orders
                SET Status = 'Shipped',
                    RowVersion = RowVersion + 1
                WHERE OrderID = 5 AND RowVersion = 3;


+-----------------------------+-----------------------------+------------------------------+
| Aspect                      | Pessimistic Concurrency     | Optimistic Concurrency       |
+-----------------------------+-----------------------------+------------------------------+
| Assumption                  | Conflicts are likely        | Conflicts are rare           |
| How it works                | Locks data until commit     | No locks; check before commit|
| Performance                 | Slower under low contention | Faster under low contention  |
| Risk                        | Deadlocks, lock contention  | Rollback & retry overhead    |
| When to use                 | High contention scenarios   | Low contention scenarios     |
| DB impact                   | More locking & blocking     | More retries but fewer locks |
| Implementation              | SELECT ... FOR UPDATE       | Row version/timestamp check  |
+-----------------------------+-----------------------------+------------------------------+

In [ ]:
# Locking Hints
1. NOLOCK
	- Equivalent to READ UNCOMMITTED
	- Doesn’t acquire locks — fastest, but dangerous
    🔴 Risk: May return dirty or inconsistent data
    ✅ Use case: Non-critical reporting queries, where accuracy is not a concern.

2. WITH (UPDLOCK, HOLDLOCK) for safe “read-then-update” patterns.
	- Used in read-then-update logic
	- Ensures no one else updates the row while you’re reading it
    ✅ Use case: Checking balance before deduction (like bank transfer)
    - Example: SELECT * FROM Accounts WITH (UPDLOCK, HOLDLOCK) WHERE AccountId = 1;
	•	WITH (UPDLOCK, HOLDLOCK) → acquires an update lock, held until transaction end.
	•	Others can still read the row (S lock is compatible with U).
	•	Others cannot update/delete the row (U lock is incompatible with U/X).


3. UPDLOCK: 
    - Prevents others from modifying the row until transaction ends.
    - Acquires update locks instead of shared locks during reads. Update locks are compatible with shared (read) locks but not with other update or exclusive locks.

4. HOLDLOCK: 
    - Usecase: Data does NOT change between read and write
    - Makes sure others can’t read uncommitted data (serializable behavior).
    - Holds shared locks until the transaction completes.
    - Prevents phantom reads by locking key ranges (not just rows).

5. ROWLOCK
	- Forces row-level locks instead of default page/table escalation
    ✅ Use case: High-concurrency systems, like stock updates for flash sales

6. XLOCK
	- Acquires exclusive lock immediately
    ✅ Use case: Prevent any read or write access by others until transaction ends

7. TABLOCKX
    - Locks the entire table exclusively
    - Example: DELETE FROM Logs WITH (TABLOCKX);

Note: Overusing locks (especially table locks) can reduce concurrency and lead to deadlocks.

In [ ]:
# Dead lock
 - If occurs when two or more clients are fighting for the same resources, and each of them are waiting for the other one to release the lock.
 - Process 1  -> waiting for  -> row 1 -> waiting for process 2 and process 2 waiting for row 2 and row 2 waiting for process 1
 - Example: Two client terminals connected with same DB.
    1. Client 1: Begin Transaction, then Try to insert row with uniquie id=10; 
    2. Client 2: Begin Transaction, then Try to insert row with uniquie id=10; 
    3. None of them has commited the transaction yet, so client 2 will be in a deadlock state, on hold/ waiting for DB response.
    4. Case 1: If we commit client 1, it will be successfull then client 2 will throw an error duplicate key exist.
    5. Case 2: If we try to insert another row in client 1, then it will throw a deadlock detected error and rollback the transaction. and Client 2 transaction can be commited.


# Two phase locking
    - First acquire all locks and in second phase when all transactions are done, then release all locks.
    - To solve, double booking problem
    - Two clients are trying to book the same seat.

Case 1: Problem: Without two phase locks
    - same seat is assigned to two persons.
        a. Client 1: Begin transaction -> check if row with id=10, seat is available - yes
        b. Client 2: Begin transaction -> check if row with id=10, seat is available - yes
        c. Client 1: Update a row, seat status =1 with his name.
        d. Client 2: Update a row, seat status =1 with his name.
        e. Client 1: commit - success operation
        f. Client 2: commit - success operation

Case 2: Solution: With locks
        a. Client 1: Begin transaction -> check if row with id=10, seat is available - yes
            - Query: select * from seats where id=10 for update;  # row with exclusive lock.                                                                      
        b. Client 2: Begin transaction -> check if row with id=10, seat is available - No result as query will not response as it is locked.
            - Query: select * from seats where id=10 for update;  # row with exclusive lock.                                                              
        c. Client 1: Update a row, seat status =1 with his name.
        e. Client 1: commit - success operation
        f. Client 2: then we will get response for client 2, with the stauts as already booked.


# Limit Offset is slower
1. Using limit and offset (Not-recomended)
    - Query: SELECT * FROM orders ORDER BY order_date LIMIT 10 OFFSET 100000;        
    - Con: 
        - The bigger the OFFSET, the slower the query.
        - We might read the same row twice, if a new row inserted in the mean time.
    - Working:
    	1.	Scan or index-scan through the first 100,000 rows.
    	2.	Throw them away.
    	3.	Return only the next 10.

2. Using filters
    - Query: SELECT *  FROM orders WHERE order_date > 2025-01-01 10:00:00 ORDER BY order_date LIMIT 10; -- Remember the last seen id/order_date from previous page
    - Working
        - Uses an index seek to jump directly to the start point.
    	- Avoids scanning and discarding large numbers of rows.
    	- Performance is O(N) for page size, not O(N+offset).

# 3. Read Phenomenas

In [ ]:

Transaction Read Phenomena:
    ├── Dirty Read → Read uncommitted data
    ├── Non-Repeatable Read → Same query gives different results (updated rows)
    ├── Phantom Read → Same query gives different row set (inserted/deleted rows)
    ├── Lost Update → Concurrent updates overwrite each other
    ├── Write Skew → Concurrent conditional updates lead to invalid states
    └── Read Skew → Inconsistent snapshot of data at different points in time



Read Phenomena (Diagrams attached in PDF):

1. Dirty Read
   - Description: A transaction reads data that another transaction has written but not yet committed.
   - Example: 
       T1 updates a row but doesn't commit.
       T2 reads the updated row.
       T1 rolls back → T2 read invalid data.

2. Non-Repeatable Read or Fuzzy Read
   - Description: A transaction reads the same row twice and gets different data because another transaction updated it in between.
   - Example: 
       T1 reads a row.
       T2 updates and commits that row.
       T1 reads the row again → sees different value.

3. Phantom Read
   - Description: A transaction runs the same query twice and gets a different set of rows because another transaction inserted or deleted rows.
   - Example: 
       T1: SELECT * FROM orders WHERE amount > 100; (can be a non-range query)
       T2 inserts a new order with amount = 200 and commits.
       T1 runs the same query again → new row appears.

4. Lost Update
    - Occurs when two concurrent transactions read the same data and then update it, but one update overwrites the other without knowing it, causing data loss.
    - Fix: Row locking
    - Example:
            T1:                      T2:
            ----                     ----
            Read Balance = 1000      Read Balance = 1000
            Balance -= 100           Balance -= 200
            Write Balance = 900      Write Balance = 800
            
            Final Balance = 800  ← Lost T1’s update!

5. Read Skew (Temporary Anomaly)
    - A transaction reads multiple related values that are individually consistent but not from the same point in time, leading to a logically inconsistent view of the data.
    - Fix: Use Snapshot Isolation 
    - Example:
            Doctor Transaction (T1):
                Read: patient_diagnosis = "Diabetes"
                Read: prescription = "No insulin"
            
                → Decision: Issue warning (inconsistent)
            
            Meanwhile...
                T2: Updates prescription to include insulin, and commits between T1’s reads.
            
            T1 has now read a **mix of old and new data**.
            
                
6. Write Skew
    - Occurs when two concurrent transactions read overlapping data and write based on what they read — and although they don’t write the same rows, their combined effect violates a constraint.
    - Neither transaction directly violates the constraint, but their combined effect does.
    - Fix: Use Serializable isolation
    - Example
        - Condition: At least one doctor must be on-call at all times
        - Table: on_call
            Rows: (DoctorA, true), (DoctorB, true)
            
            T1:                     T2:
            ----                    ----
            Read DoctorA: true      Read DoctorB: true
            Set DoctorA: false      Set DoctorB: false
            Write DoctorA: false    Write DoctorB: false
            
            Now: No doctors on call ← constraint violated!

# 4. Isolation Levels

In [ ]:
Isolation Levels
    - Used to fix the Read Phenomenas

# Isolation level
Isolation Level     | Dirty Read | Non-Repeatable Read | Phantom Read | Notes
                                    and Lost Updates
---------------------|------------|----------------------|---------------|-----------------------------------------------------------
Read Uncommitted     | ✅ Allowed | ✅ Allowed           | ✅ Allowed     | Lowest isolation, highest concurrency
Read Committed       | ❌ Prevented| ✅ Allowed           | ✅ Allowed     | Default in many RDBMS
Repeatable Read      | ❌ Prevented| ❌ Prevented         | ✅ Allowed     | Locks rows read, not range
Serializable         | ❌ Prevented| ❌ Prevented         | ❌ Prevented   | Uses range locks, may reduce concurrency
Snapshot Isolation   | ❌ Prevented| ❌ Prevented         | ❌ Prevented   | MVCC-based, may allow Write Skew

# Isolation Strength (Low → High):

Read Uncommitted
    ↓
Read Committed
    ↓
Repeatable Read
    ↓
Snapshot Isolation
    ↓
Serializable


# Example
-- Session 1
SET TRANSACTION ISOLATION LEVEL READ COMMITTED;
BEGIN TRANSACTION;

UPDATE Accounts
SET Balance = Balance - 100
WHERE AccountId = 1;

-- Simulate long operation
WAITFOR DELAY '00:00:15';


-- Session 2 (run during the delay in Session 1) (running in parallel)
SET TRANSACTION ISOLATION LEVEL READ COMMITTED;
SELECT Balance FROM Accounts WHERE AccountId = 1;

Benefits: Because we’re using READ COMMITTED, the second session waits until Session 1 commits. this prevents dirty reads — Session 2 cannot read the uncommitted deduction of ₹100.

In [ ]:
1. Read Uncommited
    - Transactions can read data written by other uncommitted transactions (i.e., dirty reads are allowed).
    - It allows dirty reads: a session can read data that another uncommitted transaction has written.
    - Use Case: Rarely used, only when absolute performance is critical and inconsistency is acceptable (e.g., logging systems).

2. Read Committed (Default in SQL Server)
    - Transactions can only read committed data. If another transaction has modified a row but not committed, it is not visible.
    - Consistency: Medium. Prevents dirty reads but still allows non-repeatable reads.
	- Use Case: Suitable for most standard CRUD operations.

3. Repeatable Read
    - Ensures that if a transaction reads a row, no other transaction can modify that row until the transaction completes. So, in the same transaction you can read 1000 times, the data will reamin same.
	- Consistency: High. Prevents dirty and non-repeatable reads.
	- Limitation: Does not prevent phantom reads (new rows may appear if the query is run again).
	- Use Case: Useful when data must not change during transaction (e.g., inventory systems).

4. Serializable
    - Transactions are executed as if they were run one after another (serially).
	- Consistency: Highest. Prevents all anomalies: dirty reads, non-repeatable reads, and phantom reads.
	- Performance: Most restrictive and can lead to high contention/locking.
	- Use Case: Financial transactions, where absolute correctness is essential.
	- Phenomena Allowed: None.

# Example of Serializable
    - Locks the read range and rows involved in the query.
    - The balance of Account 1 is locked for both read and update
    - No other transaction can even read or insert rows in the affected range (AccountId = 1) until this transaction commits
    - If two sessions run this query at the same time:
        - The second transaction will block until the first commits.
        - This ensures full isolation and serializable behavior, as if the transactions were executed one after the other.

# Query 
SET TRANSACTION ISOLATION LEVEL SERIALIZABLE;
BEGIN TRANSACTION;

-- Step 1: Check balance of Account 1
DECLARE @Balance DECIMAL(10,2);
SELECT @Balance = Balance FROM Accounts WHERE AccountId = 1;

-- Step 2: If balance is sufficient, transfer amount
IF @Balance >= 1000
BEGIN
    -- Deduct from Account 1
    UPDATE Accounts
    SET Balance = Balance - 1000
    WHERE AccountId = 1;

    -- Add to Account 2
    UPDATE Accounts
    SET Balance = Balance + 1000
    WHERE AccountId = 2;
END

COMMIT;





5. Snapshot Isolation (Non-ANSI, but common)
    - Like snapshot of complete database, achieved using versioning
	- Each transaction sees a consistent snapshot of the database as of its start time. Writes are blocked if data was changed since the snapshot.
	- Implementation: Achieved using MVCC (multi-version concurrency control).
	- Use Case: Reporting, analytics, systems where readers must not block writers.
	- Phenomena Allowed: Write skew (but not dirty/non-repeatable/phantom reads).

In [ ]:
1. When to Rely on Isolation Levels
    ✅ Use isolation levels when:
    	•	You want a consistent behavior across the entire transaction.
    	•	Your workload is typical OLTP (order entry, payments, etc.).

2. When to Use Explicit Locks
    ✅ Use explicit locks when:
    	•	You want to protect only one query instead of raising the isolation for all queries.
    	•	You need row-level update intent without blocking readers unnecessarily.
    	•	You want serialization behavior only for a critical query.
    	•	You want to avoid phantoms in just one query, not everywhere.

In [ ]:
# FAQ
1. When a transaction commits successfully but other transactions that started before it couldn't see the changes this means
    - The isolation level of the transactions started before the transaction is higher than read-committed, means transaction with repeatable read isolation will not pick up committed reads.

2. If we update and select in the same transaction, it will always fetch the updated result regardless of Isolation level.

# 5. Eventual Consistency

In [ ]:
- Eventual consistency is a consistency model used in distributed systems (especially NoSQL databases), where:
    - All nodes will become consistent over time, but not necessarily immediately after a write.
    - It may have Inconsistent reads.

- Idea
    - When you write data, it is propagated asynchronously to other replicas/nodes.
    - Reads from different nodes might temporarily return stale data.
    - Eventually, all replicas will converge to the same value.
    - It ensures
        ✅ High availability and performance
        ❌ No guarantee of immediate consistency

# When to Use Eventual Consistency?
    - High availability and partition tolerance are more important than immediate accuracy
    - Stale reads are acceptable for a short period
    - Example
        - Social media feeds, likes
        - Product catalog systems
        - Analytics counters

        
+--------------------------+----------------------------------------+---------------------------------------------+
|         Feature          |          Strong Consistency           |           Eventual Consistency             |
+--------------------------+----------------------------------------+---------------------------------------------+
| Read after Write         | Always returns the latest data        | May return stale data temporarily           |
| Consistency Guarantee    | Immediate and global                  | Eventually consistent across replicas       |
| Availability             | Lower (may reject requests)           | Higher (serves requests even during sync)   |
| Partition Tolerance      | Limited                                | High                                        |
| Latency                  | Higher (due to synchronization)       | Lower (returns immediately)                 |
| System Type              | CP (Consistency + Partition tolerant) | AP (Availability + Partition tolerant)      |
| Use Cases                | Banking, financial systems            | Social media, shopping carts, IoT data      |
| Complexity               | Simple to reason about                | Requires handling stale/duplicate data      |
| Conflict Resolution      | Not usually needed                    | May require manual or automated conflict resolution |
| Common in                | SQL databases (PostgreSQL, SQL Server)| NoSQL databases (Cassandra, DynamoDB, etc.) |
+--------------------------+----------------------------------------+---------------------------------------------+

Analogy: If we use a cache, the system is Eventual Consistent.


# 6. Connection Pooling

In [ ]:
1. Database connection (what happens when we connect)
    a. It creates a network socket (TCP, TLS/SSL) connection, authenticated via password.
    b. Server allocates session resources: Threads/ Process.
    c. Client get a session id/ context and can run statements.

2. Terminal/ CLI behaviour
	•	Tools like psql, mysql, sqlcmd open a single persistent connection (socket + server session).
	•	That connection remains open while the terminal is open; every query uses that same session (so SET values and temp tables persist).
	•	Closing the terminal or \q closes the connection and frees server resources.     
[psql client] <------TCP/TLS------> [DB Server - session S1]
   query1                                     |
   query2                                     +-- server resources (process/thread, mem)
   \q => connection closed 
                                                                                                     
3. Connection pooling 
     - A connection pool is a client‑side (or proxy) cache of physical database sessions. Instead of closing the socket and server session when your code calls Close(), the pool keeps the physical session alive and hands it to the next requester.

[Thread1] -\
[Thread2] -+-> [Connection Pool] <------> [DB Server]
[Thread3] -/           | acquires slot        (N physical sessions)
                     available connections

Flow:
	1.	Thread asks pool for a connection (acquire).
	2.	Pool gives an idle physical connection or opens a new one (if under MaxPool).
	3.	Thread uses connection (executes queries).
	4.	Thread calls .close() / .dispose() -> connection is returned to pool (not closed physically).
	5.	Pool monitors idle connections (evict after idle timeout).

# Pool Properties
	•	Min Pool Size: number of connections kept open idle.
	•	Max Pool Size: maximum physical connections allowed.
	•	Acquire Timeout: how long a request waits if pool exhausted.
	•	Validation / Health Check: test connection before handing out (optional).
	•	Idle timeout: close idle connections after N seconds.
	•	Keyed by connection string: different connection strings = different pools.


# How many connections we can maintain
	•	Depends on DB server config and memory. Typical guidelines:
    	•	Postgres default max_connections ≈ 100 (can be increased but consumes RAM per connection).
    	•	MySQL defaults vary; can handle more but still costly.
	•	Right sizing:
    	•	Web servers: pool_size ≈ number of worker threads per process. If you have 4 web server instances × 16 threads and each has pool_size=20 → total 80 physical connections.
    	•	Make sure sum(all_pools) ≤ DB max_connections (or put PgBouncer/Proxy).
    	•	If you need thousands of concurrent clients, use a proxy (transaction pooling) or scale DB horizontally with read replicas or sharding.     

# Serverless
    - For serverless, Use a managed serverless DB proxy (RDS proxy for AWS)

# Best Practices
    •	Use a connection pool for web apps — don’t open per-query.
	•	Size pool to match your app concurrency and DB max connections.
	•	Use context managers (using, with) to ensure return to pool.
	•	If using serverless, use proxy or serverless-aware pooling strategies.
	•	Monitor DB open connections (pg_stat_activity/SHOW PROCESSLIST).
	•	Ensure pool returns connections clean (auto-rollback, session reset).
	•	Use PgBouncer transaction pooling if you need to support many clients. 
    •	One connection can’t run two queries at the exact same moment unless the DB supports multiplexing (rare for traditional SQL DBs, more common in some NoSQL o                                                            

# FAQ
1. Does calling .Close() kill the TCP socket?
    - Usually no when pooling is enabled — .Close() returns connection to pool. The pool may eventually close the socket if idle timeout or pool shrink happens.

2. Why do we get ‘too many connections ?
    - Because sum of all app pools + tools + direct admin connections exceeds DB max_connections.

3. What’s the cost per connection?
    - Server allocates memory and CPU scheduling; for Postgres it’s a backend process; for MySQL a thread. Costs can be several MBs and depend on workload.

4. Can one process have multiple pools?
    - Yes — pools are keyed by connection string and driver. Two different connection strings => two pools.                                                        